In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


1-Load the training data. Report the number of observations and features. What fraction
of borrowers defaulted? Show a summary statistics table (mean, std, min, max) for all
features. Do any features need scaling before you can use distance-based methods like
KNN or SVM? Why?

In [3]:
#loading data
X_train = pd.read_csv('X_train.csv', index_col=0)
y_train = pd.read_csv('y_train.csv')

In [ ]:

#number of observations and features
print("Number of observations:", X_train.shape[0])
print("Number of features:", X_train.shape[1])
#fraction of borroweres defaulted
default_fraction = y_train['default'].sum() / len(y_train)
print("Fraction of borrowers defaulted:", default_fraction)


In [ ]:
#summary statistics table
X_train.describe()

In [ ]:
# Do any features need scaling?
# all the numerical features need scaling because they are on different scales. For example, 'loan_amount' is in the range of thousands, while 'interest_rate' is in the range of single digits. Scaling the features will help the model to converge faster and improve performance.
# all categorical features need to be encoded because machine learning models can only work with numerical data. Encoding the categorical features will allow the model to learn from them and make better predictions.

2. Fit a logistic regression to predict default. Standardize (sklearn StandardScaler)
the features first and explain why that matters for comparing coefficients. Report the
training accuracy. Which three features have the largest absolute coefficients? What
do they tell you about default risk?

In [ ]:
# standardizing features to mean 0 and standard deviation 1
scaled_X_train = X_train.copy()
scaler = StandardScaler()   
scaled_X_train[X_train.columns] = scaler.fit_transform(scaled_X_train[X_train.columns])
scaled_X_train.head()

In [ ]:
# fit a logistic regression to predict default
model = LogisticRegression(max_iter=1000)
model.fit(scaled_X_train, y_train['default'])
weights = model.coef_[0]
print("Coefficients:", weights)
#the most important features are those with the largest absolute coefficients.
# We can find the indices of the three largest absolute coefficients and then get the corresponding feature names.
feature_names = X_train.columns
abs_weights = np.abs(weights)
top_indices = np.argsort(abs_weights)[-3:]
top_features = feature_names[top_indices]
print("Top 3 features with largest absolute coefficients:", top_features)

In [ ]:
# compute the accuracy of the model
y_pred = model.predict(scaled_X_train)
accuracy = np.mean(y_pred == y_train['default'])
print("Accuracy:", accuracy)

3- Apply 𝑘-nearest neighbors with at least three values of 𝑘 (e.g., 𝑘 ∈ {5, 20, 50}). Try
both weights='uniform' and weights='distance'. Report the error rate for each
setting (you may use cross-validation or a train-test split). Which setting works best?
Why might KNN have difficulties with this dataset?

In [ ]:
# Loop over k and weights, compute cross-validated accuracy 

k_values = [5, 20, 50]
weight_options = ['uniform', 'distance']
results = []

for k in k_values:
    for w in weight_options:
        knn = KNeighborsClassifier(n_neighbors=k, weights=w)
        # Use 5-fold cross-validation
        scores = cross_val_score(knn, scaled_X_train, y_train['default'], cv=5, scoring='accuracy')

        results.append({
            "k": k,
            "weights": w,
            "Accuracy": scores.mean()
        })

df_results = pd.DataFrame(results)
print(df_results)

4. Fit a support vector machine with (a) a linear kernel and (b) an rbf kernel. For
the RBF kernel, try different values of 𝐶 (e.g., 𝐶 ∈ {0.1, 1, 10}). Compare to logistic
regression. Does the RBF kernel do better than the linear kernel? Explain what that
tells you about the shape of the decision boundary.

In [ ]:
# (a) Linear SVM 
linear_svm = SVC(kernel='linear', C=1)  # C=1 is default
linear_scores = cross_val_score(linear_svm, scaled_X_train, y_train['default'], cv=5, scoring='accuracy')
print("Linear SVM accuracy:", round(linear_scores.mean(), 2))

In [ ]:
#(b) RBF SVM with different C values
C_values = [0.1, 1, 10]
results = []

for C in C_values:
    rbf_svm = SVC(kernel='rbf', C=C, gamma='scale')  # 'scale' is recommended default
    scores = cross_val_score(rbf_svm, scaled_X_train, y_train['default'], cv=5, scoring='accuracy')
    results.append({"C": C, "Accuracy": scores.mean()})

df_rbf_results = pd.DataFrame(results)
print(df_rbf_results)

5- Pick two features and train a logistic regression and an SVM (RBF kernel) using only
those two features. Plot the decision boundaries of both classifiers side by side using
a mesh grid (see tutorial, you can re-use the code from there). What does the plot tell
you about how flexible each classifier is?

In [ ]:
# Pick two features 
features = ['previous_defaults', 'credit_utilization']
X_two = scaled_X_train[features].values

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_two, y_train['default'])

# SVM with RBF kernel
svm_rbf = SVC(kernel='rbf', C=1, gamma='scale')
svm_rbf.fit(X_two, y_train['default'])

In [ ]:
# Define grid
x_min, x_max = X_two[:, 0].min() - 1, X_two[:, 0].max() + 1
y_min, y_max = X_two[:, 1].min() - 1, X_two[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

In [ ]:
#Predict on grid
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()])
Z_lr = Z_lr.reshape(xx.shape)

Z_svm = svm_rbf.predict(np.c_[xx.ravel(), yy.ravel()])
Z_svm = Z_svm.reshape(xx.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression
axes[0].contourf(xx, yy, Z_lr, alpha=0.3, cmap=plt.cm.coolwarm)
axes[0].scatter(X_two[:, 0], X_two[:, 1], c=y_train['default'], cmap=plt.cm.coolwarm, edgecolors='k')
axes[0].set_title("Logistic Regression (linear)")
axes[0].set_xlabel(features[0])
axes[0].set_ylabel(features[1])

# SVM RBF
axes[1].contourf(xx, yy, Z_svm, alpha=0.3, cmap=plt.cm.coolwarm)
axes[1].scatter(X_two[:, 0], X_two[:, 1], c=y_train['default'], cmap=plt.cm.coolwarm, edgecolors='k')
axes[1].set_title("SVM with RBF kernel")
axes[1].set_xlabel(features[0])
axes[1].set_ylabel(features[1])

plt.show()

6-Fit a random forest. Experiment with n_estimators and max_depth. Report the
top 5 most important features (.feature_importances_). Do they match the most
important coefficients from logistic regression in (2.)? Discuss any differences.

In [ ]:

# Define parameter ranges to search over in Random Forest
n_estimators_list = [50, 100, 200]
max_depth_list = [3, 5, 10]

# Loop over all combinations
results = []

for n in n_estimators_list:
    for d in max_depth_list:
        rf = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        # 5-fold cross-validation
        scores = cross_val_score(rf, scaled_X_train, y_train['default'], cv=5, scoring='accuracy')
        mean_accuracy = scores.mean()
        results.append({"n_estimators": n, "max_depth": d, "accuracy": mean_accuracy})
        print(f"n_estimators={n}, max_depth={d}, accuracy={mean_accuracy:.4f}")

# Convert to DataFrame for easier viewing
df_results = pd.DataFrame(results)
#print("\nAll results:\n", df_results)

In [ ]:
# Take the best hyperparameters and fit the final model
best = df_results.loc[df_results['accuracy'].idxmax()]
best_rf = RandomForestClassifier(n_estimators=int(best['n_estimators']), max_depth=int(best['max_depth']), random_state=42)
best_rf.fit(scaled_X_train, y_train['default'])

# Top 5 features
importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_rf.feature_importances_
}).sort_values(by="importance", ascending=False).head(5)

print("\nTop 5 features:\n", importances)

7. Compare all your models (logistic regression, best KNN, best SVM, random forest).
a. Report the accuracy of each model. You may use cross-validation or a single
train-test split, just be consistent across models.
b. Is accuracy a good metric here? Think about the class distribution you found in
(1.) and discuss when accuracy can be misleading.

In [ ]:
# All the accuracy values are already computed above and 
# we can compare them in the paper

8. Through clever marketing campaigns, I was able to attract new potential customers who
also want a credit from my bank. I have collected the same characteristics for these new
customers, but of course I don’t know if they are creditworthy. I would like to use my
trained models to predict their default probabilities.
• New Customer Data:
– X_test.csv — 2500 × 20 feature matrix (in data_assignment/)
– y_test is not provided; you will predict these labels.
a. Using your best model from (7.) (trained on all training data), predict
the default probabilities 𝑝𝑖̂ = 𝑃(default = 1 ∣ 𝑋𝑖) on X_test.csv (using
.predict_proba() method). Save them as predictions.csv with columns
id (row index 0 to 2499) and probability. Add this csv file to your
submission.

In [ ]:
# Task 8: Final prediction using the best model from Task 7

# Read test data
X_test = pd.read_csv('data_assignment/X_test.csv', index_col=0)

# Standardize test data using the scaler fitted on training data
scaled_X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Train the best model from Task 7: SVM with RBF kernel
best_svm = SVC(kernel='rbf', C=1, gamma='scale', probability=True)
best_svm.fit(scaled_X_train, y_train['default'])

# Predict default probabilities
svm_proba = best_svm.predict_proba(scaled_X_test)[:, 1]

# Create submission file
predictions_df = pd.DataFrame({
    'id': range(len(scaled_X_test)),
    'probability': svm_proba
})

# Save to CSV
predictions_df.to_csv('predictions.csv', index=False)

print("Predictions saved to predictions.csv")
predictions_df.head()

NameError: name 'pd' is not defined